# Breeze-ASR-26 → WhisperKit Core ML Conversion

Convert MediaTek's Breeze-ASR-26 (Taiwanese Hokkien ASR) to WhisperKit Core ML format for iOS on-device deployment.

**Requirements:**
- GPU runtime (A100 40GB or H100 recommended)
- ~30-60 minutes
- ~20GB disk space

In [ ]:
# Install dependencies
!pip install -q whisperkittools coremltools torch transformers huggingface_hub tqdm

In [ ]:
# Check GPU
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Set environment variables
import os
os.environ["MODEL_REPO_ID"] = "MediaTek-Research/Breeze-ASR-26"

# Optional: HF token if needed
# os.environ["HF_TOKEN"] = "your_token_here"

In [ ]:
# Run conversion
!python -m whisperkittools.scripts.generate_model \
    --model-version MediaTek-Research/Breeze-ASR-26 \
    --output-dir ./breeze-asr-26-coreml \
    --generate-quantized-variants

In [ ]:
# Verify output
import os
from pathlib import Path

output_dir = Path("./breeze-asr-26-coreml")
required = ["AudioEncoder.mlmodelc", "TextDecoder.mlmodelc", "MelSpectrogram.mlmodelc", "tokenizer.json", "config.json"]

for f in required:
    path = output_dir / f
    exists = path.exists() or path.with_suffix("").is_dir()
    size = path.stat().st_size / 1e6 if path.exists() else 0
    print(f"{'✅' if exists else '❌'} {f} ({size:.1f} MB)")

# Check quantized variants
for d in output_dir.glob("*4bit*"):
    print(f"✅ Quantized: {d.name}")

In [ ]:
# Create zip for download
!cd breeze-asr-26-coreml && zip -r ../breeze-asr-26-coreml.zip .
print("📦 Created breeze-asr-26-coreml.zip for download")

In [ ]:
# Download link (run in Colab)
from google.colab import files
files.download("breeze-asr-26-coreml.zip")

# Optional: Upload to Hugging Face Hub

Set your HF token and repo ID below to upload the converted model.

In [ ]:
# Upload to Hugging Face Hub (optional)
# Set these before running:
# HF_TOKEN = "your_hf_token_here"
# HF_REPO = "your-username/breeze-asr-26-whisperkit-coreml"

import os
HF_TOKEN = os.environ.get("HF_TOKEN", "")
HF_REPO = os.environ.get("HF_REPO", "")

if HF_TOKEN and HF_REPO:
    from huggingface_hub import HfApi, create_repo
    api = HfApi(token=HF_TOKEN)
    create_repo(HF_REPO, token=HF_TOKEN, private=False, exist_ok=True, repo_type="model")
    api.upload_folder(
        folder_path="./breeze-asr-26-coreml",
        repo_id=HF_REPO,
        token=HF_TOKEN,
        repo_type="model",
        commit_message="Add Breeze-ASR-26 WhisperKit Core ML model (auto-converted)"
    )
    print(f"✅ Uploaded to https://huggingface.co/{HF_REPO}")
else:
    print("⚠️  Skipping HF upload (set HF_TOKEN and HF_REPO env vars to enable)")